**Assignment - Credit card defaults**

https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients

Import the dataset into a pandas dataframe. The first row consists of names that are
slightly more descriptive than just ’X1’. If you want, you can rename the column names
using the first row, then drop the first row. You can use df.rename and
df.drop(df.index[0], axis=0)  
The dataset contains a column named Unnamed: 0, which is just an index. You can throw
this column away using df.drop.

The code given on the webpage is loading the data from the web.
  
Below is code to read data from xls (extracted from zip from web)

In [26]:
import pandas as pd

df = pd.read_excel('data/default of credit card clients.xls', header=1)
# PAY_x is not consistent with PAY_AMTx
# target has long name
df.rename(columns={'PAY_0': 'PAY_1',
                   'default payment next month': 'IS_DEFAULT'
                   }, inplace=True)
# drop the first column (ID)
df.drop(columns=['ID'], inplace=True)

# split it in X and y
X = df.drop(columns=['IS_DEFAULT'])
y = df['IS_DEFAULT']

Split the dataset into a train-, and test set, using  
`train_test_split(X, y, test_size=0.5, random_state=42)`

In [27]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)

print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

(15000, 23) (15000,) (15000, 23) (15000,)


 Write a preprocessor that  
• applies a one-hot encoding to process the categorical variables.  
• rescales the features that are not categorical using a standardscale

First, check which columns have limited unique values, these are likely
to be categorical

In [45]:
print(df.nunique())

categorical_cols = [col for col in X.columns if X[col].nunique() < 10]
print(categorical_cols)

LIMIT_BAL        81
SEX               2
EDUCATION         7
MARRIAGE          4
AGE              56
PAY_1            11
PAY_2            11
PAY_3            11
PAY_4            11
PAY_5            10
PAY_6            10
BILL_AMT1     22723
BILL_AMT2     22346
BILL_AMT3     22026
BILL_AMT4     21548
BILL_AMT5     21010
BILL_AMT6     20604
PAY_AMT1       7943
PAY_AMT2       7899
PAY_AMT3       7518
PAY_AMT4       6937
PAY_AMT5       6897
PAY_AMT6       6939
IS_DEFAULT        2
dtype: int64
['SEX', 'EDUCATION', 'MARRIAGE']


So, from SEX, EDUCATION and MARRIAGE the one-hot encoding can be done

In [51]:
from sklearn.preprocessing import OneHotEncoder


def applyonehot(X, colname):
  col_cat = X[[colname]]
  onehot_encoder = OneHotEncoder(sparse_output=False, drop='first')
  col_encoded = onehot_encoder.fit_transform(col_cat)
  col_df = pd.DataFrame(col_encoded, columns=onehot_encoder.get_feature_names_out([colname]))
  X = X.reset_index(drop=True)
  X_return = pd.concat([X, col_df], axis=1).drop(colname, axis=1)
  return X_return

categorical_cols = [col for col in X.columns if X[col].nunique() < 10]
for col in categorical_cols:
  X_train = applyonehot(X_train, col)

print(X_train.head())


   LIMIT_BAL  AGE  PAY_1  PAY_2  PAY_3  PAY_4  PAY_5  PAY_6  BILL_AMT1  \
0      80000   47     -2     -2     -2     -2     -2     -1     194843   
1     220000   35     -1     -1     -1     -1     -1     -1       8304   
2     300000   33     -1     -1     -2     -2     -2     -2      76922   
3      80000   53      0      0      0      0      0      0      31333   
4     500000   32      0     -1     -1     -1      0      0      13631   

   BILL_AMT2  ...  SEX_2  EDUCATION_1  EDUCATION_2  EDUCATION_3  EDUCATION_4  \
0     197581  ...    1.0          0.0          0.0          1.0          0.0   
1      11526  ...    1.0          1.0          0.0          0.0          0.0   
2       -694  ...    1.0          0.0          0.0          1.0          0.0   
3      32365  ...    1.0          0.0          1.0          0.0          0.0   
4       3840  ...    1.0          0.0          1.0          0.0          0.0   

   EDUCATION_5  EDUCATION_6  MARRIAGE_1  MARRIAGE_2  MARRIAGE_3  
0       